# D5 Class-Level Pixel Motif Graph Retrieval on Kaggle

Pipeline: CSV -> graph_repo -> D5A train -> evaluate -> visualize motif masks.

## Required Kaggle Input

- FER-2013 split CSV dataset containing `train.csv`, `val.csv`, `test.csv`

Optional:

- Prebuilt `graph_repo` dataset if `BUILD_GRAPH=False`

In [ ]:
import os
import subprocess
from pathlib import Path
from pathlib import Path

GITHUB_USERNAME = "YOUR_USERNAME"
GITHUB_REPO_NAME = "YOUR_REPO"
GITHUB_REPO_BRANCH = "main"
WORK_DIR = Path("/kaggle/working")
REPO_PATH = WORK_DIR / GITHUB_REPO_NAME
GH_TOKEN = os.environ.get("GH_TOKEN", "")
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")

if not REPO_PATH.exists():
    if GH_TOKEN:
        url = f"https://{GH_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO_NAME}.git"
    else:
        url = f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO_NAME}.git"
    subprocess.run(["git", "clone", "--branch", GITHUB_REPO_BRANCH, url, str(REPO_PATH)], check=True)

PROJECT_PATH = REPO_PATH / "fer_d5"
if not PROJECT_PATH.exists():
    PROJECT_PATH = REPO_PATH
os.chdir(PROJECT_PATH)
print("Project:", PROJECT_PATH)

subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY

In [ ]:
EXPERIMENT_CONFIG = "configs/d5a_kaggle.yaml"

RUN_SMOKE = True
RUN_FULL_ONE_EPOCH = False
RUN_FULL_TRAIN = False
RUN_VISUALIZE = True
CSV_ROOT = "auto"
GRAPH_REPO_INPUT_PATH = None
BATCH_SIZE_OVERRIDE = None
USE_WANDB = False
ZIP_OUTPUTS = True

MODE = "smoke"
EPOCHS_OVERRIDE = None
MAX_TRAIN_BATCHES = 3
MAX_VAL_BATCHES = 2
MAX_TEST_BATCHES = 2

if RUN_FULL_ONE_EPOCH:
    RUN_SMOKE = False
    MODE = "build_and_train"
    EPOCHS_OVERRIDE = 1
    MAX_TRAIN_BATCHES = None
    MAX_VAL_BATCHES = None
    MAX_TEST_BATCHES = None
elif RUN_FULL_TRAIN:
    RUN_SMOKE = False
    MODE = "build_and_train"
    EPOCHS_OVERRIDE = None
    MAX_TRAIN_BATCHES = None
    MAX_VAL_BATCHES = None
    MAX_TEST_BATCHES = None

In [ ]:
import subprocess

cmd = ["python", "scripts/run_experiment.py", "--config", EXPERIMENT_CONFIG, "--mode", MODE]
if CSV_ROOT:
    cmd += ["--csv_root", CSV_ROOT]
if GRAPH_REPO_INPUT_PATH:
    cmd += ["--graph_repo_path", GRAPH_REPO_INPUT_PATH]
if BATCH_SIZE_OVERRIDE is not None:
    cmd += ["--batch_size", str(BATCH_SIZE_OVERRIDE)]
if EPOCHS_OVERRIDE is not None:
    cmd += ["--epochs", str(EPOCHS_OVERRIDE)]
if MAX_TRAIN_BATCHES is not None:
    cmd += ["--max_train_batches", str(MAX_TRAIN_BATCHES)]
if MAX_VAL_BATCHES is not None:
    cmd += ["--max_val_batches", str(MAX_VAL_BATCHES)]
if MAX_TEST_BATCHES is not None:
    cmd += ["--max_test_batches", str(MAX_TEST_BATCHES)]
if not USE_WANDB:
    cmd += ["--no_wandb"]
if ZIP_OUTPUTS:
    cmd += ["--zip_outputs"]

print(" ".join(cmd))
subprocess.run(cmd, check=True)

if RUN_VISUALIZE and MODE in {"smoke", "train"}:
    checkpoint = Path("/kaggle/working/fer_d5_outputs/checkpoints/best.pth")
    if checkpoint.exists():
        viz_cmd = ["python", "scripts/run_experiment.py", "--config", EXPERIMENT_CONFIG, "--mode", "visualize", "--checkpoint", str(checkpoint)]
        if GRAPH_REPO_INPUT_PATH:
            viz_cmd += ["--graph_repo_path", GRAPH_REPO_INPUT_PATH]
        if BATCH_SIZE_OVERRIDE is not None:
            viz_cmd += ["--batch_size", str(BATCH_SIZE_OVERRIDE)]
        if not USE_WANDB:
            viz_cmd += ["--no_wandb"]
        print(" ".join(viz_cmd))
        subprocess.run(viz_cmd, check=True)

In [ ]:
from pathlib import Path

for path in [Path('/kaggle/working/artifacts/graph_repo'), Path('/kaggle/working/fer_d5_outputs')]:
    print('\n', path)
    if path.exists():
        for child in sorted(path.glob('*'))[:30]:
            print(' ', child)

manifest = Path('/kaggle/working/artifacts/graph_repo/manifest.pt')
checkpoints = Path('/kaggle/working/fer_d5_outputs/checkpoints')
figures = Path('/kaggle/working/fer_d5_outputs/figures')
print('manifest exists:', manifest.exists())
print('checkpoints:', list(checkpoints.glob('*')) if checkpoints.exists() else [])
print('figures:', list(figures.glob('*')) if figures.exists() else [])

In [ ]:
from pathlib import Path
from IPython.display import Image, display

output_root = Path('/kaggle/working/fer_d5_outputs')
cm = output_root / 'evaluation' / 'confusion_matrix.png'
if cm.exists():
    display(Image(filename=str(cm)))

for p in sorted((output_root / 'figures' / 'd5a_class_gates').glob('*.png'))[:7]:
    display(Image(filename=str(p)))

for p in sorted((output_root / 'figures' / 'd5a_attention').rglob('*.png'))[:8]:
    display(Image(filename=str(p)))

In [ ]:
from pathlib import Path
import zipfile

if ZIP_OUTPUTS:
    output_root = Path('/kaggle/working/fer_d5_outputs')
    zip_path = Path('/kaggle/working/fer_d5_outputs_manual.zip')
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for p in output_root.rglob('*'):
            if p.is_file():
                zf.write(p, p.relative_to(output_root.parent))
    print(zip_path)